#  Collision Resolver
> Decide how two vocabularies can coexist inside a JSON-LD document.

 Resolution order  
1. **Registry hint** – `registry[a].strategy_defaults.get(b)`  
2. **Bundled table** – `data/collision_data.json` (legacy)  
3. Default → `separate_graphs`

The caller (e.g. 04_Composer) gets back a `Plan` object.


## Collision Resolver: “Safety‑First” Design

This document explains the rationale, algorithm and API of the safety‑first collision‑handling layer introduced in 03_collision.  It is meant for contributors who need to understand why the resolver behaves the way it does, and how to extend or override its logic.

⸻

1  Background

JSON‑LD 1.1 allows a term to be marked "@protected": true. Once protected, that term must not be re‑defined inside the same active context; an attempt to do so raises an error during expansion.  When two vocabularies are merged naïvely, collisions on protected terms are therefore a direct source of runtime failures.

Earlier versions of CogitareLink relied on a static lookup table (collision_data.json) to specify how well‑known vocabulary pairs (e.g. VC + EPCIS) should be combined.  Unknown pairs defaulted to separate_graphs, which is safe but often over‑kill.

The new resolver adds an adaptive safety layer so that any pair of vocabularies—whether pre‑registered or not—can be merged without breaking JSON‑LD rules.

⸻

2  Decision Ladder

Priority	Condition	Strategy	Notes
R1	Registry hint exists (either direction)	use hint	Human‑curated, explicit
R2	Bundled legacy rule exists	use rule	Maintains backwards‑compat
R3	Both vocabs declare @protected and they overlap	separate_graphs	Zero risk of redefine‑errors
R4	One vocab declares @protected or no overlap between sets	nested_contexts (outer = protected)	Keeps protected term stable; inner vocab lives in distinct context scope
R5	Neither vocab declares protection	separate_graphs (default)	Still safe, simplest merge

Overlap is defined as an intersection between the two sets of protected term keys.

The ladder is applied after checking (a == b) which always yields separate_graphs.

⸻

3  Implementation Sketch
```python
# private helpers
@lru_cache(maxsize=128)
def _protected_terms(ctx):
    if "@context" in ctx: ctx = ctx["@context"]
    return {t for t,d in ctx.items()
            if isinstance(d, dict) and d.get("@protected")}

def _overlap(a, b):
    ea, eb = registry[a], registry[b]
    sa = _protected_terms(ea.context_payload())
    sb = _protected_terms(eb.context_payload())
    return bool(sa), bool(sb), bool(sa & sb)

# inside Resolver.choose()
row = self._registry_hint(a,b) or self._bundled(a,b)
if not row:
    a_has, b_has, overlap = _overlap(a,b)
    if a_has or b_has:
        if a_has and b_has and overlap:
            return Plan(strategy=Strategy.separate_graphs,
                        details={"reason":"overlapping @protected"})
        outer, inner = (a,b) if a_has else (b,a)
        return Plan(strategy=Strategy.nested_contexts,
                    details={"outer":outer,"inner":inner})
    row = dict(strategy="separate_graphs")
return Plan(strategy=Strategy(row["strategy"]), details=row)
```
All expensive operations (context fetch + term extraction) are cached, so the overhead is negligible after first use.

⸻

4  Practical Outcomes
- Zero configuration safety for any new vocab the user adds at runtime.
- Composer can treat nested_contexts and separate_graphs identically to before—no code changes needed.
- Existing curated rules still work and override the heuristics.

⸻

5  Extending
    1.	Add a curated rule: edit entry.strategy_defaults in registry_data.json or collision_data.json.  Curated rules always rank above heuristics.
    2.	Disable the heuristic for a pair: add a rule with "strategy":"graph_partition" or another explicit value.
    3.	Change the safety policy: patch _overlap or inject a different dynamic checker—you do not need to touch Composer.

⸻

6  Glossary

Term	Meaning
protected term	A context key whose definition includes "@protected": true.
outer/inner context	In a nested_contexts plan, the outer context becomes the top‑level @context, while the inner one is embedded under a designated property.
graph partition	Strategy where each vocab’s triples live in a distinct named graph; out of scope for this heuristic layer.

In [ ]:
#| default_exp vocab.collision

##  1 – imports  

In [ ]:
#| export
from __future__ import annotations

import json, importlib.resources as pkg
from enum import Enum
from functools import lru_cache
from typing import Any, Dict, Optional

from pydantic import BaseModel

from cogitarelink.core.debug import get_logger
from cogitarelink.vocab.registry import registry, preferred_collision

log = get_logger("collision")

In [ ]:
#| hide
# bundled JSON (copy of the old COLLISION_STRATEGIES)
_TABLE: Dict[str, Any] = json.loads(
    pkg.files("cogitarelink").joinpath("data/collision_data.json").read_text()
)
_TABLE

{"('vc', 'epcis')": {'strategy': 'property_scoped',
  'primary': 'vc',
  'property': 'credentialSubject',
  'secondary': 'epcis',
  'description': 'Places EPCIS data within the VC credentialSubject property'},
 "('schema', 'dc')": {'strategy': 'graph_partition',
  'primary': 'schema',
  'description': 'Partitions the graph with Schema.org as the primary vocabulary'},
 "('schema', 'foaf')": {'strategy': 'property_mapping',
  'mappings': {'foaf:name': 'schema:name',
   'foaf:Person': 'schema:Person',
   'foaf:knows': 'schema:knows'},
  'description': 'Maps FOAF properties to Schema.org equivalents'},
 "('ro-crate', 'dcat')": {'strategy': 'nested_contexts',
  'outer': 'ro-crate',
  'inner': 'dcat',
  'description': 'Nests DCAT context within RO-Crate'},
 "('prov', 'schema')": {'strategy': 'context_versioning',
  'context_version': '1.1',
  'description': 'Uses JSON-LD 1.1 features to handle these vocabularies'},
 "('shacl', 'owl')": {'strategy': 'separate_graphs',
  'description': 'Mainta

## %% 2 – load bundled table  

In [ ]:
#| export
_BUNDLED: Dict[str, Dict[str, Any]] = json.loads(
    pkg.files("cogitarelink").joinpath("data/collision_data.json").read_text()
)
# keys are "(a,b)" strings – keep legacy layout

## %% 3 – datatypes 

In [ ]:
#| export

class Strategy(str, Enum):
    property_scoped   = "property_scoped"
    graph_partition   = "graph_partition"
    nested_contexts   = "nested_contexts"
    context_versioning= "context_versioning"
    separate_graphs   = "separate_graphs"


class Plan(BaseModel):
    strategy: Strategy
    details:  Dict[str, Any] = {}

    model_config = dict(frozen=True)

## %% 4 – protected-term helpers 

In [ ]:
#| export
 
def _protected_terms(ctx: Dict[str, Any]) -> set[str]:
    "Return the set of keys whose definition contains `\"@protected\": true`."
    if "@context" in ctx:
        ctx = ctx["@context"]
    return {
        term
        for term, defi in ctx.items()
        if isinstance(defi, dict) and defi.get("@protected") is True
    }

In [ ]:
#| export
@lru_cache(maxsize=128)
def _prot_overlap(a: str, b: str) -> tuple[bool, bool, bool]:
    """
    Cached check returning:
        (a_has_protected, b_has_protected, overlap_exists)
    """
    # Check if terms exist in registry
    a_exists = a in registry._v
    b_exists = b in registry._v
    
    # If either doesn't exist, they can't have protected terms
    if not a_exists or not b_exists:
        return False, False, False
        
    ea, eb = registry[a], registry[b]
    sa = _protected_terms(ea.context_payload())
    sb = _protected_terms(eb.context_payload())
    return bool(sa), bool(sb), bool(sa & sb)


## %% 5 – resolver  

In [ ]:
#| export
class Resolver:
    "Figure out _how_ two vocabularies should be merged."

    # -------------------------------- private helpers --------------------
    def _registry_hint(self, a: str, b: str) -> Optional[Dict[str, Any]]:
        "Look for a strategy hinted in either vocab's `strategy_defaults`."
        return preferred_collision(a, b) or preferred_collision(b, a)

    def _bundled(self, a: str, b: str) -> Optional[Dict[str, Any]]:
        "Lookup (a,b) or (b,a) in the legacy table."
        key1 = f"('{a}', '{b}')"
        key2 = f"('{b}', '{a}')"
        return _BUNDLED.get(key1) or _BUNDLED.get(key2)
    
    # -------------------------------- public API ------------------------
    def choose(self, a: str, b: str) -> Plan:
        """
        Return a **Plan** describing how vocabularies *a* and *b* should
        coexist in the same JSON-LD document.
        """
        # identical vocabularies never collide
        if a == b:
            return Plan(strategy=Strategy.separate_graphs)

        # 1 – explicit registry hint
        row = self._registry_hint(a, b)
        if row:
            log.debug(f"registry hint ({a},{b}) → {row}")
            return Plan(strategy=Strategy(row["strategy"]),
                        details={k: v for k, v in row.items() if k != "strategy"})

        # 2 – bundled rule
        row = self._bundled(a, b)
        if row:
            log.debug(f"bundled rule ({a},{b}) → {row}")
            return Plan(strategy=Strategy(row["strategy"]),
                        details={k: v for k, v in row.items() if k != "strategy"})

        # 3 – dynamic safety heuristic
        a_has, b_has, overlap = _prot_overlap(a, b)
        if a_has or b_has:                           # at least one uses @protected
            if a_has and b_has and overlap:
                log.debug(f"protected-overlap ({a},{b}) → separate_graphs")
                return Plan(strategy=Strategy.separate_graphs,
                            details={"reason": "overlapping @protected terms"})
            outer, inner = (a, b) if a_has else (b, a)
            log.debug(f"protected one-way ({a},{b}) → nested_contexts")
            return Plan(strategy=Strategy.nested_contexts,
                        details={"outer": outer, "inner": inner})

        # 4 – safe default
        log.debug(f"default ({a},{b}) → separate_graphs")
        return Plan(strategy=Strategy.separate_graphs)

## %% 6 – module-level singleton  


In [ ]:
#| export
resolver = Resolver()

In [ ]:
#| hide
#%% 7 – quick tests  
from fastcore.test import test_eq, test_ne

# 7.1 bundled VC + EPCIS -> property_scoped
p1 = resolver.choose("vc", "epcis")
test_eq(p1.strategy, Strategy.property_scoped)

# 7.2 identical prefixes -> separate_graphs
p2 = resolver.choose("schema", "schema")
test_eq(p2.strategy, Strategy.separate_graphs)

# 7.3 unknown pair with no protection -> default
p3 = resolver.choose("foo", "bar")
test_eq(p3.strategy, Strategy.separate_graphs)

# 7.4 synthetic protected vs open vocab (heuristic)
dummy_prot_ctx = {"@context":{"z":{"@id":"x:z","@protected":True}}}
dummy_open_ctx = {"@context":{"z":"x:z2"}}

# monkey-patch two temporary registry entries
# Use absolute import instead of relative
from cogitarelink.vocab.registry import VocabEntry, ContextBlock, Versions
registry._v["prot"] = VocabEntry(
    prefix="prot",
    uris={"primary":"http://ex/"},
    context=ContextBlock(inline=dummy_prot_ctx),
    versions=Versions(current="1.0")
)
registry._v["open"] = VocabEntry(
    prefix="open",
    uris={"primary":"http://ex2/"},
    context=ContextBlock(inline=dummy_open_ctx),
    versions=Versions(current="1.0")
)
p4 = resolver.choose("prot", "open")
test_eq(p4.strategy, Strategy.nested_contexts)
test_eq(p4.details["outer"], "prot")
test_ne(p4.details["outer"], p4.details["inner"])

In [ ]:
#| hide
# Test case where both vocabs have @protected terms with overlap
registry._v["prot2"] = VocabEntry(
    prefix="prot2",
    uris={"primary":"http://ex3/"},
    context=ContextBlock(inline={"@context":{"z":{"@id":"x:z3","@protected":True}}}),
    versions=Versions(current="1.0")
)
p5 = resolver.choose("prot", "prot2")
test_eq(p5.strategy, Strategy.separate_graphs)
test_eq(p5.details.get("reason"), "overlapping @protected terms")

# Test registry hint (if we can mock one)
# This would verify priority level R1 in your decision ladder

# Test with a non-existent vocabulary
p6 = resolver.choose("nonexistent", "open")
test_eq(p6.strategy, Strategy.separate_graphs)


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()